In [0]:
from datetime import datetime, timedelta

# FHIR API configuration
BASE_URL = "https://hapi.fhir.org/baseR4"

FHIR_RESOURCES = [
    "Patient",
    "Encounter",
    "Observation",
    "Condition"
]

PAGE_SIZE = 100

# 3-day ingestion window: today + previous 2 days
END_DATE = datetime.now().date()
START_DATE = END_DATE - timedelta(days=2)

# Exclusive API boundary so today's complete data is included
API_END_DATE = END_DATE + timedelta(days=1)

In [0]:
import requests
import json
import os
import time
from datetime import datetime, timezone


RAW_BASE_PATH = "/Volumes/fhir_assignment/raw/fhir_files"


def ingest_fhir_resource(
    resource,
    start_date,
    end_date,
    api_end_date,
    page_size=100
):
    """
    Incrementally ingest a FHIR resource.

    Handles:
    - _lastUpdated incremental filtering
    - FHIR pagination
    - Raw JSON response storage
    - Extraction and save timestamps
    - API URL/parameters metadata
    """

    # Build the initial API URL
    url = (
    f"{BASE_URL}/{resource}"
    f"?_lastUpdated=ge{start_date}T00:00:00"
    f"&_lastUpdated=lt{api_end_date}T00:00:00"
    f"&_count={page_size}"
)

    # Metadata for this ingestion run
    extraction_timestamp = datetime.now(timezone.utc).isoformat()

    page_number = 1
    total_records = 0

    # Raw folder structure
    raw_path = (
        f"{RAW_BASE_PATH}/"
        f"{resource}/"
        f"{start_date}_{end_date}"
    )

    os.makedirs(raw_path, exist_ok=True)

    while url:

        # Record exactly when this API call happens
        api_call_timestamp = datetime.now(timezone.utc).isoformat()

        print(f"Fetching {resource} - page {page_number}")

        response = requests.get(url, timeout=60)
        response.raise_for_status()

        # Keep the original API response
        bundle = response.json()

        records_in_page = len(bundle.get("entry", []))
        total_records += records_in_page

        # Metadata about this particular API call
        metadata = {
            "resource": resource,
            "page_number": page_number,
            "extraction_timestamp": extraction_timestamp,
            "api_call_timestamp": api_call_timestamp,
            "api_url_or_params": url,
            "records_in_page": records_in_page,
            "data_saved_timestamp": datetime.now(timezone.utc).isoformat()
        }

        # Save original FHIR response
        file_name = f"page_{page_number}.json"
        file_path = f"{raw_path}/{file_name}"

        with open(file_path, "w") as f:
            json.dump(bundle, f)

        # Save metadata alongside the response
        metadata_file = f"{raw_path}/page_{page_number}_metadata.json"

        with open(metadata_file, "w") as f:
            json.dump(metadata, f, indent=2)

        print(f"  Records in page: {records_in_page}")

        # Find the next page from the FHIR Bundle
        next_url = None

        for link in bundle.get("link", []):
            if link.get("relation") == "next":
                next_url = link.get("url")
                break

        url = next_url
        page_number += 1

        # Small delay between API requests
        time.sleep(0.2)

    print(f"\n{resource} ingestion complete")
    print(f"Total records: {total_records}")
    print(f"Raw location: {raw_path}")

    return {
        "resource": resource,
        "total_records": total_records,
        "extraction_timestamp": extraction_timestamp,
        "raw_path": raw_path
    }

In [0]:
patient_result = ingest_fhir_resource(
    resource="Patient",
    start_date=START_DATE,
    end_date=END_DATE,
    api_end_date=API_END_DATE,
    page_size=PAGE_SIZE
)

encounter_result = ingest_fhir_resource(
    resource="Encounter",
    start_date=START_DATE,
    end_date=END_DATE,
    api_end_date=API_END_DATE,
    page_size=PAGE_SIZE
)

observation_result = ingest_fhir_resource(
    resource="Observation",
    start_date=START_DATE,
    end_date=END_DATE,
    api_end_date=API_END_DATE,
    page_size=PAGE_SIZE
)

condition_result = ingest_fhir_resource(
    resource="Condition",
    start_date=START_DATE,
    end_date=END_DATE,
    api_end_date=API_END_DATE,
    page_size=PAGE_SIZE
)